# Preprocessing — EFC_compositional

Builds the two files every later analysis reads:

| file | one row per | contents |
|---|---|---|
| `s<sn>.csv` | trial | the cleaned `.dat` plus `sn`, `day`, and `MD` |
| `s<sn>_mov.pkl` | trial | `t`, `state`, the gain-corrected smoothed `force`, and `force_raw` (unsmoothed) |

Order matters: the `.pkl` is built first, then `MD` is computed from those
smoothed traces and written back into the `.csv`.

Run `sanity_check.ipynb` before this.

## Imports

In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# project root = the folder that contains target/ and data/
ROOT = Path.cwd()
while not (ROOT / "target").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.design import day_of_block

SUBJECT = 104

# Session structure (utils/design.py): which day each block belongs to. Filenames are
# built from this rather than discovered, so the naming convention is enforced, not assumed.
DAY_OF_BLOCK = day_of_block(SUBJECT)

MOV_NAME = "EFC_compositional_{sn}_{bn:02d}.mov"

DAT_PATH = ROOT / "data" / f"s{SUBJECT}" / f"EFC_compositional_{SUBJECT}.dat"
MOV_DIR = DAT_PATH.parent          # data/ holds raw data only
TGT_DIR = ROOT / "target"

ANALYSIS_DIR = ROOT / "analysis"   # processed, subject-level dataframes live here
ANALYSIS_DIR.mkdir(exist_ok=True)
OUT_CSV = ANALYSIS_DIR / f"s{SUBJECT}.csv"
OUT_PKL = ANALYSIS_DIR / f"s{SUBJECT}_mov.pkl"

FS = 500          # Hz, sampling rate of the .mov (2 ms per sample)
WAIT_EXEC = 3     # state code for the execution phase, see EFC_compositional.h

print(ROOT)

from utils.movload import movload


/Users/aghavamp/Desktop/Projects/EFC_compositional


## 1. Trial table from the `.dat`

Loads the `.dat`, adds `sn` and `day`. `MD` is created empty here and filled in later after the .mov files are processed.

In [5]:
df = pd.read_csv(DAT_PATH, sep="\t")
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]   # trailing tab on every row
df.columns = df.columns.str.strip()

df.insert(0, "sn", SUBJECT)
df.insert(1, "day", df["BN"].map(DAY_OF_BLOCK))
df["MD"] = np.nan

COL_ORDER = (["sn", "day", "BN", "TN", "chordID", "trialPoint", "planError", "RT", "ET", "MD"]
             + ["planTime", "success_holdTime", "max_holdTime", "execMaxTime", "feedbackTime", "iti"]
             + ["forceGain", "fGain1", "fGain2", "fGain3", "fGain4", "fGain5"]
             + ["verticalShift", "baselineTopThresh", "extTopThresh", "extBotThresh",
                "flexTopThresh", "flexBotThresh"]
             + ["subNum", "DateTime"])
df = df[[c for c in COL_ORDER if c in df.columns]
        + [c for c in df.columns if c not in COL_ORDER]]

# dupes is used to grab the trial row from the .mov dataframe
dupes = df[df.duplicated(subset=["BN", "TN"], keep=False)]
if len(dupes):
    print("!" * 78)
    print(f"WARNING: {len(dupes)} rows share a (BN, TN) pair in blocks "
          f"{sorted(dupes['BN'].unique().tolist())}.")
    print("Sections 2-3 index .mov trials by TN, so those rows will be matched to the")
    print("wrong force traces. Run sanity_check.ipynb and clean the .dat before trusting")
    print("anything below.")
    print("!" * 78)

print(f"{len(df)} trials, {df['BN'].nunique()} blocks, "
      f"days {sorted(int(d) for d in df['day'].unique())}")
df.head(3)


1040 trials, 52 blocks, days [1, 2, 3, 4]


,sn,day,BN,TN,chordID,trialPoint,planError,RT,ET,MD,...,fGain4,fGain5,verticalShift,baselineTopThresh,extTopThresh,extBotThresh,flexTopThresh,flexBotThresh,subNum,DateTime
0,104,1,1,1,21912,0,0,1214.0,0.0,NaN,...,1.5,1.5,0,1.2,5,2,-5,-2,104,"30-09-2026,13:44:13"
1,104,1,1,2,21912,0,0,388.0,0.0,NaN,...,1.5,1.5,0,1.2,5,2,-5,-2,104,"30-09-2026,13:44:27"
2,104,1,1,3,21921,0,1,2.0,0.0,NaN,...,1.5,1.5,0,1.2,5,2,-5,-2,104,"30-09-2026,13:44:40"


## 2. Get force traces from the `.mov` files

The differential force in the `.mov` is raw (`ext - flex`, no gains), so it is
multiplied here by the per-trial `forceGain` and `fGain1..5` (thumb → pinky) taken
from the `.dat`.

`force_raw` keeps the same gain-applied force *unsmoothed* (the `.mov`'s visualised
force column): it is the exact signal the task thresholds, so event times (RT, acquisition,
a single finger leaving baseline, see `utils/events.py`) are detected on it, while
kinematic measures such as MD use the smoothed `force`.

Smoothing is a 4th-order Butterworth zero phase low-pass. One can set `METHOD = "boxcar"` for an edge-padded moving average smoothing - was just testing this tbh. Use the lowpass imo.

In [6]:
SMOOTH_MS = 40          # <<< force smoothing time constant in ms (30-50 sensible here)
METHOD = "butter"       # <<< "butter" = zero-phase low-pass, "boxcar" = moving average

# a SMOOTH_MS boxcar and a 1/(2*SMOOTH_MS) Hz low-pass suppress roughly the same band
CUTOFF_HZ = 1000 / (2 * SMOOTH_MS)

if METHOD == "butter":
    b, a = butter(4, CUTOFF_HZ / (FS / 2), btype="low")
    smooth = lambda x: filtfilt(b, a, x, axis=0)          # filtfilt -> no phase lag
    print(f"smoothing: 4th-order Butterworth low-pass at {CUTOFF_HZ:.1f} Hz (filtfilt)")
else:
    k = max(1, int(round(SMOOTH_MS / 1000 * FS)))
    k += (k % 2 == 0)                                     # odd -> window stays centred
    kernel = np.ones(k) / k
    # edge-padded so the ends are not dragged toward zero
    smooth = lambda x: np.column_stack(
        [np.convolve(np.pad(x[:, i], k // 2, mode="edge"), kernel, mode="valid")
         for i in range(x.shape[1])])
    print(f"smoothing: {SMOOTH_MS} ms moving average -> {k} samples at {FS} Hz")

FGAIN_COLS = ["fGain1", "fGain2", "fGain3", "fGain4", "fGain5"]
rows = []

for bn in sorted(df["BN"].unique()):
    mov_path = MOV_DIR / MOV_NAME.format(sn=SUBJECT, bn=bn)
    trials, cols = movload(mov_path)
    blk = df[df["BN"] == bn]

    if len(trials) < blk["TN"].max():
        raise ValueError(f"{mov_path.name}: {len(trials)} trials but the .dat "
                         f"expects up to TN {blk['TN'].max()}")

    for _, tr in blk.iterrows():
        mov = trials[int(tr["TN"]) - 1]

        gg = float(tr["forceGain"])
        fg = tr[FGAIN_COLS].to_numpy(float)
        raw = mov[:, cols["diff"]]

        # the task's own visualised force is the same signal with gains applied,
        # so this confirms both the column mapping and the gains from the .dat
        assert np.abs(mov[:, cols["vis"]] - gg * fg * raw).max() < 1e-2, \
            f"gain mismatch in {mov_path.name} trial {int(tr['TN'])}"

        force = smooth(raw * fg * gg)

        rows.append(dict(
            sn=int(tr["sn"]), day=int(tr["day"]), BN=int(tr["BN"]), TN=int(tr["TN"]),
            chordID=int(tr["chordID"]),
            t=mov[:, cols["time"]].copy(),
            state=mov[:, cols["state"]].astype(int),
            globalGain=gg,
            fingerGain=fg,
            force=force,
            force_raw=mov[:, cols["vis"]].copy(),   # gains applied, unsmoothed: what the task saw
        ))

mov_df = pd.DataFrame(rows, columns=["sn", "day", "BN", "TN", "chordID",
                                     "t", "state", "globalGain", "fingerGain", "force", "force_raw"])
print(f"\n{len(mov_df)} trials, "
      f"{sum(r.shape[0] for r in mov_df['force']):,} samples total")
mov_df.head(3)


smoothing: 4th-order Butterworth low-pass at 12.5 Hz (filtfilt)

1040 trials, 3,736,955 samples total


,sn,day,BN,TN,chordID,t,state,globalGain,fingerGain,force,force_raw
0,104,1,1,1,21912,"[0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...",1.0,"[1.0, 1.0, 1.0, 1.5, 1.5]","[[1.6200309945322071, -0.7995045188718091, 0.1...","[[1.615, -0.801, 0.141, 0.191, -0.021], [1.63,..."
1,104,1,1,2,21912,"[0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...",1.0,"[1.0, 1.0, 1.0, 1.5, 1.5]","[[-0.7404333765085629, 0.7615470784548372, 0.5...","[[-0.74, 0.761, 0.511, 0.049, -0.174], [-0.739..."
2,104,1,1,3,21921,"[0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16...","[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, ...",1.0,"[1.0, 1.0, 1.0, 1.5, 1.5]","[[1.016664923690548, -0.6466142171212169, -1.0...","[[1.013, -0.646, -1.092, -0.609, -0.535], [1.0..."


## 3. Mean deviation

`MD` measures how simultaneous a chord is: the average distance of the 5-finger
force trajectory from the straight line joining its start and end points
(Waters-Metenier et al. 2014). A perfectly simultaneous chord moves straight to the
target and scores 0; one that moves fingers in sequence scores high.

The window runs from the go cue plus `RT` to the moment the chord was acquired
(`RT + ET`), both taken from the `.dat`, with the execution phase located via
`state == WAIT_EXEC`. Only correct trials get an `MD` others nan.

In [7]:
record = {(r.BN, r.TN): r for r in mov_df.itertuples()}
md_values = np.full(len(df), np.nan)
skipped = 0

for i, tr in df.iterrows():
    if tr["trialPoint"] != 1: # no acquisition -> no endpoint -> MD undefined
        continue

    rec = record[(int(tr["BN"]), int(tr["TN"]))]
    in_exec = rec.state == WAIT_EXEC
    if not in_exec.any():
        skipped += 1
        continue

    t0 = rec.t[in_exec][0] # go cue
    window = (rec.t >= t0 + tr["RT"]) & (rec.t <= t0 + tr["RT"] + tr["ET"])
    F = rec.force[window]
    if len(F) < 3:
        skipped += 1
        continue

    c = F[-1] - F[0] # ideal straight-line path
    if c @ c == 0:
        skipped += 1
        continue

    d = F - F[0]
    projection = np.outer(d @ c / (c @ c), c)
    md_values[i] = np.linalg.norm(d - projection, axis=1)[1:].mean()

df["MD"] = md_values

n_ok = int((df["trialPoint"] == 1).sum())
print(f"MD computed for {int(np.isfinite(md_values).sum())} of {n_ok} correct trials"
      + (f" ({skipped} skipped: window too short)" if skipped else ""))
print(df.groupby("chordID")["MD"].agg(["count", "mean", "median"]).round(3))


MD computed for 941 of 941 correct trials
         count   mean  median
chordID                      
12912       89  1.959   1.908
12921       88  1.734   1.773
12999      296  0.201   0.079
21912       94  1.945   1.823
21921       83  1.969   1.885
99912      291  0.578   0.476


## 4. Save

In [8]:
df.to_csv(OUT_CSV, index=False)
mov_df.to_pickle(OUT_PKL)

print(f"{OUT_CSV.relative_to(ROOT)}   {len(df)} rows x {df.shape[1]} cols  "
      f"({OUT_CSV.stat().st_size / 1e3:.0f} KB)")
print(f"{OUT_PKL.relative_to(ROOT)}   {len(mov_df)} rows  "
      f"({OUT_PKL.stat().st_size / 1e6:.1f} MB)")

analysis/s104.csv   1040 rows x 30 cols  (151 KB)
analysis/s104_mov.pkl   1040 rows  (359.0 MB)
